![image](car.jpeg)

**Car-ing is sharing**, an auto dealership company for car sales and rental, is taking their services to the next level thanks to **Large Language Models (LLMs)**.

As their newly recruited AI and NLP developer, you've been asked to prototype a chatbot app with multiple functionalities that not only assist customers but also provide support to human agents in the company.

The solution should receive textual prompts and use a variety of pre-trained Hugging Face LLMs to respond to a series of tasks, e.g. classifying the sentiment in a car’s text review, answering a customer question, summarizing or translating text, etc.


In [27]:
# Import necessary packages
import pandas as pd
import torch

from transformers import logging
logging.set_verbosity(logging.WARNING)

In [28]:
# Start your code here!
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline
from datasets import load_dataset

import evaluate 
df=pd.read_csv('data/car_reviews.csv', delimiter=';')

reviews= df['Review'].tolist()
labels= df['Class'].tolist()

#sentimentAnalysis = pipeline("text-classification", model='tabularisai/multilingual-sentiment-analysis')
sentimentAnalysis = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')

predicted_labels = sentimentAnalysis(reviews)
print(predicted_labels)

#for review, prediction, label in zip(reviews, predicted_labels, labels):
    #print(f"review: {review},  prediction:{prediction['label']}, label: {label}")

predictions = [1 if pred['label']=='Positive' else 0 for pred in predicted_labels]
labels_num = [1 if label=='POSITIVE' else 0 for label in labels]

accuracy = evaluate.load('accuracy')
accuracy_results_dict = accuracy.compute(predictions=predictions,references=labels_num)
accuracy_result = accuracy_results_dict['accuracy']
f1 = evaluate.load('f1')
f1_results_dict = f1.compute(predictions=predictions,references=labels_num)
f1_result = f1_results_dict['f1']


#Translation
sample = reviews[0]
translator = pipeline("translation", model ="Helsinki-NLP/opus-mt-en-es")
translated_review =  translator(sample)[0]['translation_text']
with open("data/reference_translations.txt", 'r') as file:
    lines = file.readlines()

references = [line.strip('\n') for line in lines]
print(references)

bleu =evaluate.load('bleu')
bleu_score = bleu.compute(predictions =[translated_review], references = [references])
#bleu_score = bleu_results['bleu']

#extrative QA
question = "What did he like about the brand?"
context = reviews[1]
model = AutoModelForQuestionAnswering.from_pretrained("deepset/minilm-uncased-squad2")
tokenizer = AutoTokenizer.from_pretrained("deepset/minilm-uncased-squad2")
tokenized_input = tokenizer(question, context, return_tensors='pt')
with torch.no_grad():
    outputs= model(**tokenized_input)
    

start_idx = torch.argmax(outputs.start_logits)
end_idx = torch.argmax(outputs.end_logits)+1


answer_span = tokenized_input['input_ids'][0][start_idx:end_idx]
answer = tokenizer.decode(answer_span)
print(f"Answer : {answer}")
#summarizer

sample= reviews[-1]
summarizer = pipeline('summarization', model ="facebook/bart-large-cnn")
summarized_text = summarizer(sample, min_length=50, max_length=55)[0]['summary_text']
print(f"Summarized Text: {summarized_text}")


Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.929397702217102}, {'label': 'POSITIVE', 'score': 0.8654273152351379}, {'label': 'POSITIVE', 'score': 0.9994640946388245}, {'label': 'NEGATIVE', 'score': 0.9935314059257507}, {'label': 'POSITIVE', 'score': 0.9986565113067627}]


Device set to use cpu


['Estoy muy satisfecho con mi Nissan NV SL 2014. Utilizo esta camioneta para mis entregas comerciales y uso personal.', 'Estoy muy satisfecho con mi Nissan NV SL 2014. Uso esta furgoneta para mis entregas comerciales y uso personal.']


Some weights of the model checkpoint at deepset/minilm-uncased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Answer : ride quality, reliability


Device set to use cpu


Summarized Text: The Nissan Rogue provides me with the desired SUV experience without burdening me with an exorbitant payment. Handling and styling are great; I have hauled 12 bags of mulch in the back with the seats down and could have held more. The engine delivers strong
